# NLinear failure-risk model: predict `fail_next_30d`

**Goal:** for each producing well-day *t*, estimate the probability that a failure starts within the next 30 days.

**Model:** NLinear, adapted for classification. The input is the last `LOOKBACK` days of every feature channel. Each window has its last value subtracted, so the model sees only the shape of the recent history. A linear layer then maps each channel's time dimension to `D` values, the last value is added back, and a linear head combines all channels into one logit. Every parameter, including two small embeddings for field and pump make, is trained by backpropagation with AdamW.

**Run:** use the `Python (datathon-2026)` kernel. The inputs are built by `src/clean.py` and `src/merge.py`.

## 1. Configuration: time split with purge gaps

In [ ]:
import json, time
import numpy as np, pandas as pd, torch, torch.nn as nn, matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name in ("src", "notebooks") else Path.cwd()
P, FIG, MODELS = ROOT / "data/processed", ROOT / "reports/figures", ROOT / "models"
FIG.mkdir(parents=True, exist_ok=True); MODELS.mkdir(exist_ok=True)

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); torch.set_num_threads(4)

TARGET = "fail_next_30d"
HORIZON = 30                  # days the label looks ahead
LOOKBACK = 30                 # days of history per input window
D = 8                         # NLinear output length per channel (the "pred_len" of the original model)

LABEL_START = pd.Timestamp("2025-06-01")   # failures are only recorded from here (see src/merge.py)
TEST_START  = pd.Timestamp("2026-06-27")   # same cutoff as src/clean.py split
VAL_START   = pd.Timestamp("2026-04-01")
GAP = pd.Timedelta(days=HORIZON)
# Purge: a label at day t covers (t, t+30]. Training labels must finish before validation starts,
# and validation labels before test starts. Otherwise the later period's failures leak into earlier labels.
TRAIN_END = VAL_START - GAP - pd.Timedelta(days=1)
VAL_END   = TEST_START - GAP - pd.Timedelta(days=1)
MAX_LOOKBACK = 60             # longest window tried in section 9; every variant uses the same samples
DATA_START = LABEL_START - pd.Timedelta(days=MAX_LOOKBACK + 35)   # room for 30-day rolling features + windows

print(f"train  {LABEL_START.date()} .. {TRAIN_END.date()}   (label windows end by {(TRAIN_END + GAP).date()})")
print(f"val    {VAL_START.date()} .. {VAL_END.date()}   (label windows end by {(VAL_END + GAP).date()})")
print(f"test   {TEST_START.date()} .. end of data;  held-out wells are excluded from train and val entirely")

## 2. Leakage guard
These columns must never be used as inputs. Some are the label itself. Others are only known because a failure happened, or describe the repair job that follows it:

| Column | Why it is excluded |
|---|---|
| `fail_next_30d`, `fail_next_7d`, `days_to_next_failure` | The target, or derived from future failures |
| `failure_active`, `failure_onset`, `active_failure_type`, `active_failure_cause` | Describe the failure itself. `failure_active` is used only to drop rows, never as an input |
| `job_active` | A job in progress is usually the repair of a failure that has already started |
| `last_job_duration_days`, `job_cost_prior` | Only known once a job finishes, and the last job may still be running on day *t* |
| `label_observed`, `split` | Bookkeeping |

The remaining job history (job counts and days since the last job) counts jobs by start date strictly before *t*. A repair job always starts on or after its failure's onset (see `src/merge.py`), so the repair of a failure the model is trying to predict can never appear in that failure's input window.

In [ ]:
LEAKY = ["fail_next_30d", "fail_next_7d", "days_to_next_failure", "failure_active", "failure_onset",
         "active_failure_type", "active_failure_cause", "job_active", "last_job_duration_days",
         "job_cost_prior", "label_observed", "split"]

## 3. Load data
Operating values come from `dashboard.parquet`, which has no gap interpolation. `model.parquet` interpolates short gaps using the next observed value, and that next value is in the future relative to day *t*. Here gaps are filled forward only.

In [ ]:
since = [("production_date", ">=", DATA_START.to_pydatetime())]
HIST = ["fail_count_prior", "fail_count_90d", "fail_count_365d", "fail_count_prior_rods", "fail_count_prior_pump",
        "fail_count_prior_tubing", "days_since_last_failure", "job_count_prior", "job_count_90d", "job_count_365d",
        "job_count_prior_rod_truck", "job_count_prior_workover", "job_count_prior_pump_change",
        "days_since_last_job", "days_since_last_rod_truck", "days_since_last_workover", "days_since_last_pump_change"]
STATIC = ["well_age_years", "surface_latitude", "surface_longitude"]
CATS = ["field_operating_area", "pcp_pump_make"]
m = pd.read_parquet(P / "merged.parquet", filters=since,
                    columns=["uwi", "production_date", "split", "well_daily_status", TARGET, "failure_active"]
                    + HIST + STATIC + CATS)

DAILY = ["hours_on_prod", "hours_down", "oil_produced_m3", "water_produced_m3", "gas_produced_e3m3",
         "tubing_pressure_kpa", "casing_pressure_kpa", "pump_efficiency_pct"]
SETPOINT = ["pcp_pump_rpm", "pcp_pump_rod_torque_ftlb", "pcp_pump_electric_current_a",
            "pcp_pump_volume_throughput_m3d", "pcp_pump_displacement_m3d100rpm"]
ops = pd.read_parquet(P / "dashboard.parquet", filters=since, columns=["uwi", "production_date"] + DAILY + SETPOINT)
m = m.merge(ops, on=["uwi", "production_date"], how="left"); del ops
m = m.sort_values(["uwi", "production_date"]).reset_index(drop=True)
num = DAILY + SETPOINT + HIST + STATIC
m[num] = m[num].astype("float32")
assert not set(LEAKY) & set(num)
print(m.shape, f"{m.memory_usage(deep=True).sum() / 1e6:.0f} MB")

## 4. Feature engineering (past-only)
- **Forward fill** within each well: up to 3 days for daily measurements, and up to 60 days for RPM, torque, amps, throughput and displacement. Those five change on only 2–9% of days, so they behave like settings that stay in place until someone updates them.
- **Rolling standard deviation over 7 days:** instability just before a failure.
- **Short-term vs long-term:** 3-day mean against 30-day mean, and 7-day mean against 30-day mean. This flags sudden changes in torque, amps, RPM, efficiency and volumes.
- **Reading-present flags:** whether amps, torque and efficiency were actually recorded in the last 30 days, so a filled-in value can be told apart from a real one.

Rolling values are computed down the whole sorted column, then any row whose window would cross into the previous well is set to NaN.

In [ ]:
g = m.groupby("uwi", sort=False)
pos = g.cumcount().to_numpy()

for c, n in [("pcp_pump_electric_current_a", "amps"), ("pcp_pump_rod_torque_ftlb", "torque"), ("pump_efficiency_pct", "eff")]:
    m[f"has_{n}"] = m[c].notna().astype("float32")
m[DAILY] = g[DAILY].ffill(limit=3)
m[SETPOINT] = g[SETPOINT].ffill(limit=60)

def roll(s, w, fn, minp, shift=0):
    r = getattr(s.shift(shift).rolling(w, min_periods=minp), fn)().to_numpy(dtype="float32", copy=True)
    r[pos < w - 1 + shift] = np.nan          # window would reach into the previous well
    return r

ENG = []
for n in ["amps", "torque", "eff"]:
    m[f"has_{n}_30d"] = roll(m[f"has_{n}"], 30, "max", 1); ENG.append(f"has_{n}_30d")
m = m.drop(columns=["has_amps", "has_torque", "has_eff"])

base = {"torque": "pcp_pump_rod_torque_ftlb", "amps": "pcp_pump_electric_current_a", "rpm": "pcp_pump_rpm",
        "eff": "pump_efficiency_pct", "oil": "oil_produced_m3", "water": "water_produced_m3",
        "hours": "hours_on_prod", "throughput": "pcp_pump_volume_throughput_m3d"}
bounded = {"eff", "hours"}        # compare as a difference; the rest as a relative change
for n, c in base.items():
    s = m[c]
    m[f"{n}_std_7d"] = roll(s, 7, "std", 4); ENG.append(f"{n}_std_7d")
    m30 = roll(s, 30, "mean", 10)
    for k in ([3, 7] if n in ("torque", "amps", "rpm") else [3]):
        mk = roll(s, k, "mean", max(1, k - 1))
        m[f"{n}_{k}v30"] = mk - m30 if n in bounded else np.where(m30 > 0, mk / m30 - 1, np.nan).astype("float32")
        ENG.append(f"{n}_{k}v30")

w7, o7 = roll(m.water_produced_m3, 7, "sum", 4), roll(m.oil_produced_m3, 7, "sum", 4)
m["water_cut_7d"] = np.where(w7 + o7 > 0, w7 / (w7 + o7), np.nan).astype("float32")
m["downtime_days_30d"] = roll((m.hours_on_prod == 0).astype("float32"), 30, "sum", 20)
ENG += ["water_cut_7d", "downtime_days_30d"]

for c in [c for c in HIST if c.startswith("days_since")]:
    m[c] = m[c].fillna(730).clip(upper=730)          # never / more than 2 years ago -> 730
CHANNELS = DAILY + SETPOINT + ENG + HIST + STATIC
assert not set(LEAKY) & set(CHANNELS)
print(len(CHANNELS), "channels:", CHANNELS)

## 5. Sample selection, split, and scaling (fit on train rows only)

**Usable day:** a day *t* becomes a sample when all of these hold:
- its label is known
- the well is producing and not already in a failure
- hours on production, oil and water are present on every day of the longest window tried (60 days), so no window has a long gap or crosses into another well

**Wells:** the 237 held-out wells are used only in the test set.

In [ ]:
CORE = ["hours_on_prod", "oil_produced_m3", "water_produced_m3"]
core_gap = m[CORE].isna().any(axis=1).astype("float32")
window_ok = roll(core_gap, MAX_LOOKBACK, "sum", MAX_LOOKBACK) == 0   # also guarantees windows never cross wells

date, held = m.production_date, m.split.eq("test_well").to_numpy(dtype=bool, na_value=False)
usable = (m[TARGET].notna() & m.failure_active.eq(0) & m.well_daily_status.eq("Producing")).to_numpy(dtype=bool, na_value=False) & window_ok
in_range = lambda a, b: ((date >= a) & (date <= b)).to_numpy()
idx = {
    "train":         np.flatnonzero(usable & ~held & in_range(LABEL_START, TRAIN_END)),
    "val":           np.flatnonzero(usable & ~held & in_range(VAL_START, VAL_END)),
    "test (seen wells)":   np.flatnonzero(usable & ~held & (date >= TEST_START).to_numpy()),
    "test (unseen wells)": np.flatnonzero(usable & held & (date >= TEST_START).to_numpy()),
}
y_all = m[TARGET].fillna(0).to_numpy("float32")
summary = pd.DataFrame({k: {"samples": len(v), "wells": m.uwi.iloc[v].nunique(), "positive rate": y_all[v].mean(),
                            "first day": date.iloc[v].min().date(), "last day": date.iloc[v].max().date()}
                        for k, v in idx.items()}).T
assert set(m.uwi.iloc[idx["train"]]).isdisjoint(m.uwi.iloc[idx["test (unseen wells)"]])
assert date.iloc[idx["train"]].max() + GAP < VAL_START and date.iloc[idx["val"]].max() + GAP < TEST_START
summary

In [ ]:
# Robust scaling with statistics from training-period rows of training wells only
LOG1P = [c for c in CHANNELS if c in ("oil_produced_m3", "water_produced_m3", "gas_produced_e3m3", "tubing_pressure_kpa",
         "casing_pressure_kpa", "pcp_pump_volume_throughput_m3d", "pcp_pump_rod_torque_ftlb",
         "pcp_pump_electric_current_a") or c.startswith(("fail_count", "job_count"))]
fit_rows = (~held) & (date <= TRAIN_END).to_numpy()
meta = m[["uwi", "production_date", "fail_count_365d"]].copy()
vocab = {c: {v: i + 1 for i, v in enumerate(sorted(m.loc[fit_rows, c].dropna().unique()))} for c in CATS}  # 0 = unknown
cat_codes = np.stack([m[c].map(vocab[c]).fillna(0).astype("int64").to_numpy() for c in CATS], axis=1)
scaler = {}
X = np.empty((len(m), len(CHANNELS)), dtype="float32")
for j, c in enumerate(CHANNELS):
    v = m[c].to_numpy("float32")
    if c in LOG1P: v = np.log1p(np.clip(v, 0, None))
    ref = v[fit_rows]; ref = ref[~np.isnan(ref)]
    med, iqr = np.percentile(ref, 50), np.subtract(*np.percentile(ref, [75, 25]))
    sc = iqr if iqr > 0 else (ref.std() if ref.std() > 0 else 1.0)
    scaler[c] = {"log1p": c in LOG1P, "median": float(med), "scale": float(sc)}
    X[:, j] = np.nan_to_num(np.clip((v - med) / sc, -10, 10), nan=0.0)   # missing -> typical (0); has_* flags tell the model
    del m[c]                                                                # free memory as we go

del m; import gc; gc.collect()
print("X", X.shape, f"{X.nbytes / 1e6:.0f} MB")

## 6. NLinear classifier

In [ ]:
class NLinearClassifier(nn.Module):
    """NLinear backbone + linear classification head.

    x: (B, L, C). Each channel's window is normalised by its last value (the NLinear trick against
    level shifts), mapped L -> D by a linear layer shared across channels (individual=False, as in
    the original NLinear), then the last value is added back so the head still sees the level.
    """
    def __init__(self, L, C, d=8, n_cats=(), emb_dim=4, individual=False, dropout=0.1):
        super().__init__()
        self.individual = individual
        self.temporal = nn.ModuleList([nn.Linear(L, d) for _ in range(C)]) if individual else nn.Linear(L, d)
        self.embs = nn.ModuleList([nn.Embedding(n + 1, emb_dim) for n in n_cats])
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(C * d + emb_dim * len(n_cats), 1))

    def forward(self, x, cats):
        last = x[:, -1:, :]                                  # (B, 1, C)
        z = (x - last).transpose(1, 2)                       # (B, C, L)
        if self.individual:
            z = torch.stack([lin(z[:, i]) for i, lin in enumerate(self.temporal)], dim=1)
        else:
            z = self.temporal(z)                             # (B, C, D)
        z = z + last.transpose(1, 2)
        h = torch.cat([z.flatten(1)] + [e(cats[:, i]) for i, e in enumerate(self.embs)], dim=1)
        return self.head(h).squeeze(-1)                      # logit

X_t, C_t, Y_t = torch.from_numpy(X), torch.from_numpy(cat_codes), torch.from_numpy(y_all)

def get_batch(rows, L):
    rows = torch.as_tensor(rows)
    win = rows[:, None] + torch.arange(-L + 1, 1)          # (B, L) row indices; windows never cross wells (window_ok)
    return X_t[win], C_t[rows], Y_t[rows]

## 7. Metrics (no scikit-learn in this environment)

In [ ]:
def roc_auc(y, p):
    r = pd.Series(p).rank().to_numpy(); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def avg_precision(y, p):
    o = np.argsort(-p, kind="stable"); ys = y[o]
    prec = np.cumsum(ys) / np.arange(1, len(ys) + 1)
    return (prec * ys).sum() / ys.sum()

def best_f1_threshold(y, p):
    o = np.argsort(-p, kind="stable"); ys, ps = y[o], p[o]
    tp = np.cumsum(ys); prec = tp / np.arange(1, len(ys) + 1); rec = tp / ys.sum()
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-9)
    k = f1.argmax(); return ps[k], f1[k]

def report(y, p, thr):
    pred = p >= thr; tp = (pred & (y == 1)).sum()
    top = np.argsort(-p)[: max(1, len(p) // 20)]
    return {"samples": len(y), "positive rate": y.mean(), "ROC-AUC": roc_auc(y, p), "PR-AUC (AP)": avg_precision(y, p),
            "AP lift vs random": avg_precision(y, p) / y.mean(),
            "precision @ threshold": tp / max(pred.sum(), 1), "recall @ threshold": tp / y.sum(),
            "precision in top 5% risk": y[top].mean()}

## 8. Training: backpropagation with AdamW, early stopping on validation PR-AUC
- **Loss:** binary cross-entropy, with `pos_weight` set from the training class ratio to handle the imbalance (about 12% positives).
- **Order:** training windows are shuffled each epoch. Each window is still in time order internally, and the time split above keeps the future out of training.

In [ ]:
def predict(model, rows, L, bs=8192):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(rows), bs):
            xb, cb, _ = get_batch(rows[i:i + bs], L)
            out.append(torch.sigmoid(model(xb, cb)).numpy())
    return np.concatenate(out)

def train(L=LOOKBACK, d=D, individual=False, epochs=40, bs=2048, lr=2e-3, wd=1e-4, patience=5, verbose=True):
    torch.manual_seed(SEED)
    model = NLinearClassifier(L, X.shape[1], d, n_cats=[len(vocab[c]) for c in CATS], individual=individual)
    ytr = y_all[idx["train"]]
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - ytr.mean()) / ytr.mean()))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    rng = np.random.default_rng(SEED)
    hist, best, best_state, wait = [], -1, None, 0
    yv = y_all[idx["val"]]
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); tot = 0.0
        order = rng.permutation(idx["train"])
        for i in range(0, len(order), bs):
            xb, cb, yb = get_batch(order[i:i + bs], L)
            loss = loss_fn(model(xb, cb), yb)
            opt.zero_grad(); loss.backward(); opt.step()        # gradient descent step on all parameters
            tot += loss.item() * len(yb)
        pv = predict(model, idx["val"], L)
        with torch.no_grad():
            vl = loss_fn(torch.logit(torch.from_numpy(pv).clamp(1e-6, 1 - 1e-6)), torch.from_numpy(yv)).item()
        ap, auc = avg_precision(yv, pv), roc_auc(yv, pv)
        hist.append({"epoch": ep, "train loss": tot / len(order), "val loss": vl, "val AP": ap, "val AUC": auc})
        if verbose: print(f"epoch {ep:2d}  train {tot / len(order):.4f}  val {vl:.4f}  val AP {ap:.3f}  AUC {auc:.3f}  ({time.time() - t0:.1f}s)")
        if ap > best: best, wait, best_state = ap, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist)

model, hist = train()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(hist.epoch, hist["train loss"], color="#2a78d6", label="train")
axes[0].plot(hist.epoch, hist["val loss"], color="#eb6834", label="validation")
axes[0].set_title("Weighted BCE loss", loc="left"); axes[0].legend(frameon=False); axes[0].set_xlabel("epoch")
axes[1].plot(hist.epoch, hist["val AP"], color="#2a78d6")
axes[1].axhline(y_all[idx["val"]].mean(), color="#52514e", ls="--", lw=1)
axes[1].set_title("Validation PR-AUC (dashed = positive rate)", loc="left"); axes[1].set_xlabel("epoch")
for ax in axes: ax.grid(color="#e4e3df"); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(FIG / "model_training.png", dpi=150); plt.show()

## 9. Lookback and channel-mixing variants (selected on validation only)
Each variant is compared on validation PR-AUC. The test set is not looked at for any of these choices.

In [ ]:
variants = {"L=30, shared": model}
scores = {"L=30, shared": hist["val AP"].max()}
for name, kw in {"L=14, shared": dict(L=14), "L=60, shared": dict(L=60), "L=30, individual": dict(individual=True)}.items():
    mdl, h = train(verbose=False, **kw)
    variants[name], scores[name] = mdl, h["val AP"].max()
    print(f"{name:18s} best val AP {scores[name]:.3f}  ({len(h)} epochs)")
best_name = max(scores, key=scores.get)
model = variants[best_name]
BEST_L = {"L=14, shared": 14, "L=60, shared": 60}.get(best_name, LOOKBACK)
print("selected:", best_name)

## 10. Baselines and final evaluation
- **Rule:** rank days by failures in the past 365 days. This was the strongest single feature in the EDA.
- **Last-day linear:** the same model with a 1-day window. It shows whether using the 30-day history helps at all.

The decision threshold is chosen on validation (maximum F1) and then applied unchanged to both test sets.

In [ ]:
lin1, _ = train(L=1, verbose=False)
val_p = predict(model, idx["val"], BEST_L)
thr, f1 = best_f1_threshold(y_all[idx["val"]], val_p)
print(f"threshold {thr:.3f} (val F1 {f1:.3f})")

rows = {}
for split in ["val", "test (seen wells)", "test (unseen wells)"]:
    r, y = idx[split], y_all[idx[split]]
    rows[(split, f"NLinear ({best_name})")] = report(y, predict(model, r, BEST_L), thr)
    p1 = predict(lin1, r, 1)
    rows[(split, "last-day linear")] = report(y, p1, best_f1_threshold(y_all[idx["val"]], predict(lin1, idx["val"], 1))[0])
    rule = meta.fail_count_365d.to_numpy()[r] + 1e-6 * np.random.default_rng(SEED).random(len(r))
    rows[(split, "rule: failures last 365d")] = {k: v for k, v in report(y, rule, np.inf).items()
                                                  if "threshold" not in k}
results = pd.DataFrame(rows).T
results.style.format({"samples": "{:,.0f}", "positive rate": "{:.1%}", "ROC-AUC": "{:.3f}", "PR-AUC (AP)": "{:.3f}",
                      "AP lift vs random": "{:.2f}x", "precision @ threshold": "{:.1%}", "recall @ threshold": "{:.1%}",
                      "precision in top 5% risk": "{:.1%}"}, na_rep="–")

## 11. Which channels the model relies on
Permutation importance on validation: shuffle one channel's whole window across samples and measure how much PR-AUC drops.

In [ ]:
rng = np.random.default_rng(SEED)
sub = rng.choice(idx["val"], size=min(30_000, len(idx["val"])), replace=False)
ysub = y_all[sub]
base_ap = avg_precision(ysub, predict(model, sub, BEST_L))
perm = rng.permutation(len(sub))
imp = {}
model.eval()
with torch.no_grad():
    for j, c in enumerate(CHANNELS):
        ps = []
        for i in range(0, len(sub), 8192):
            xb, cb, _ = get_batch(sub[i:i + 8192], BEST_L)
            xp, _, _ = get_batch(sub[perm[i:i + 8192]], BEST_L)
            xb[:, :, j] = xp[:, :, j]
            ps.append(torch.sigmoid(model(xb, cb)).numpy())
        imp[c] = base_ap - avg_precision(ysub, np.concatenate(ps))
imp = pd.Series(imp).sort_values(ascending=False)
top = imp.head(20)[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top.values, color="#2a78d6", height=0.6)
ax.set_title(f"Top 20 channels: drop in validation PR-AUC when shuffled (base {base_ap:.3f})", loc="left", fontsize=10)
ax.grid(axis="x", color="#e4e3df"); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(FIG / "model_channel_importance.png", dpi=150); plt.show()
new = [c for c in ENG if "_std_7d" in c or "v30" in c]
print("new instability / trend channels:\n", imp[new].sort_values(ascending=False).round(4).to_string())

## 12. Save the model and the test predictions

In [ ]:
torch.save({"state_dict": model.state_dict(), "channels": CHANNELS, "cats": CATS, "vocab": vocab, "scaler": scaler,
            "lookback": BEST_L, "d": D, "individual": best_name.endswith("individual"), "threshold": float(thr),
            "splits": {"train_end": str(TRAIN_END.date()), "val": [str(VAL_START.date()), str(VAL_END.date())],
                       "test_start": str(TEST_START.date())}},
           MODELS / "nlinear_fail30.pt")
out = []
for split in ["test (seen wells)", "test (unseen wells)"]:
    r = idx[split]
    out.append(meta.iloc[r][["uwi", "production_date"]].assign(split=split, risk=predict(model, r, BEST_L), fail_next_30d=y_all[r]))
pd.concat(out).to_parquet(P / "predictions_test.parquet", index=False)
print("saved models/nlinear_fail30.pt and data/processed/predictions_test.parquet")

## Results

| | ROC-AUC | PR-AUC | Precision in the top 5% riskiest days |
|---|---|---|---|
| **NLinear, test on seen wells** (Jun–Oct 2026) | 0.772 | 0.348 (2.6× random) | 46% |
| **NLinear, test on unseen wells** (237 wells never trained on) | 0.778 | 0.349 (3.2× random) | 44% |
| Rule: failures in the last 365 days, seen wells | 0.716 | 0.296 | 42% |

**What the numbers say:**
- **Better than the simple rule.** NLinear beats it on every split, and it holds up on wells it has never seen, so it isn't just memorising wells.
- **The 30-day history adds nothing yet.** The 1-day linear baseline scores the same as NLinear (test AP 0.349 vs 0.348). Most of the signal is in the level of each feature on day *t*: water volume, repair history, RPM. The shape of the last 30 days adds little. This matches the EDA: RPM, torque, amps and throughput change on only 2–9% of days, and efficiency drops only about 2% before a failure.
- **Of the new instability and trend channels, only `hours_std_7d` helps much** (−0.013 PR-AUC when shuffled). Most of the torque, amps and RPM std and 3-vs-30 channels are close to zero, because those inputs rarely change from day to day.
- **Window length and channel mixing don't matter.** L = 14, 30 and 60, and shared vs per-channel weights, all land at validation AP 0.334–0.337.
- **Threshold:** 0.647, chosen on validation. On seen-well test days it flags about half of the days that lead to a failure (52% recall), and about 1 in 3 flagged days is followed by a failure within 30 days (33% precision).
